<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/04.%203D%20Pose%20Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - 3D Pose Estimation (Bayesian MAP)**
> MARC - Multidisciplinary AI Research Centre | UOP


## **🛠️ Setup OpenDatasetStore**

In [ ]:
%%capture
# @title Store Initialization
from google.colab import drive
import pandas as pd
import numpy as np
import os
!pip install -q pandas scipy plotly pyarrow

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install -q git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"
store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local",
    entity_id_format="sub_{num:04d}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)

import plotly.io as pio
pio.renderers.default = "colab"

In [ ]:
# @title Dataset Summary
store.summary()
store.list_entries(entry_type="experiments")

# **📐 Bayesian MAP 3-D Pose Fusion Pipeline**
Fuses synchronized **Front ($X, Y$)** and **Side ($Z, Y$)** 2-D keypoint streams into a temporally smooth, anatomically consistent 3-D skeletal sequence $X \in \mathbb{R}^{T \times 17 \times 3}$.

In [ ]:
# @title Bayesian MAP 3-D Solver & Utilities
import time
import threading
import numpy as np
import pandas as pd
from scipy import signal
from scipy.sparse import lil_matrix
from scipy.optimize import least_squares
import plotly.graph_objects as go
from plotly.subplots import make_subplots

store_lock = threading.Lock()

# COCO 17 Keypoint Names
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

# COCO 17 Skeleton Pairs (P = 16)
SKELETON_PAIRS = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16)
]

BONE_NAMES = [
    f"{KEYPOINT_NAMES[p1]}_to_{KEYPOINT_NAMES[p2]}" for p1, p2 in SKELETON_PAIRS
]

# Reference long lower-body bones (hip-knee, knee-ankle) for scale normalization
REF_BONES = [(11, 13), (12, 14), (13, 15), (14, 16)]


def align_keypoint_streams(front_df, side_df, side_lag_frames):
    """
    Aligns front_df and side_df using the stored lag:
        side_lag_frames = t_front - t_side
    - If side_lag_frames > 0: Front started earlier -> drop first `side_lag_frames` from front_df.
    - If side_lag_frames < 0: Side started earlier  -> drop first `|side_lag_frames|` from side_df.
    """
    lag = int(round(side_lag_frames))
    if lag > 0:
        f_sync = front_df.iloc[lag:].reset_index(drop=True)
        s_sync = side_df.reset_index(drop=True)
    elif lag < 0:
        f_sync = front_df.reset_index(drop=True)
        s_sync = side_df.iloc[abs(lag):].reset_index(drop=True)
    else:
        f_sync = front_df.reset_index(drop=True)
        s_sync = side_df.reset_index(drop=True)

    T = min(len(f_sync), len(s_sync))
    f_sync = f_sync.iloc[:T].copy()
    s_sync = s_sync.iloc[:T].copy()
    return f_sync, s_sync


def _df_to_arrays(df):
    """
    Extracts (T, 17, 2) coordinates and (T, 17) confidence scores.
    Flipping image Y-axis (v -> -v) is optional; here we keep image-space orientation
    so reprojection matches pixel coordinates directly.
    """
    x_cols = [f"{name}_x" for name in KEYPOINT_NAMES]
    y_cols = [f"{name}_y" for name in KEYPOINT_NAMES]
    s_cols = [f"{name}_score" for name in KEYPOINT_NAMES]

    kx = df[x_cols].to_numpy(dtype=np.float64)
    ky = df[y_cols].to_numpy(dtype=np.float64)
    ks = df[s_cols].fillna(0.0).to_numpy(dtype=np.float64)

    # Mask out any residual (0,0) unpredicted coordinates
    zero_mask = (kx == 0.0) & (ky == 0.0)
    kx[zero_mask] = np.nan
    ky[zero_mask] = np.nan
    ks[zero_mask] = 0.0

    pts = np.stack([kx, ky], axis=-1)  # (T, 17, 2)
    return pts, ks


def _center_and_smooth_root(pts, scores, min_hip_conf=0.1):
    """
    Computes the per-frame pelvic root (midpoint of left_hip=11 and right_hip=12),
    smooths the root trajectory temporally to prevent rigid-body hip jitter,
    and subtracts the smoothed root from all joints.
    """
    T, J, _ = pts.shape
    raw_roots = np.full((T, 2), np.nan, dtype=np.float64)

    for t in range(T):
        c11, c12 = scores[t, 11], scores[t, 12]
        p11, p12 = pts[t, 11], pts[t, 12]
        valid_11 = (c11 >= min_hip_conf) and not np.any(np.isnan(p11))
        valid_12 = (c12 >= min_hip_conf) and not np.any(np.isnan(p12))

        if valid_11 and valid_12:
            raw_roots[t] = (c11 * p11 + c12 * p12) / (c11 + c12)
        elif valid_11:
            raw_roots[t] = p11
        elif valid_12:
            raw_roots[t] = p12
        else:
            valid_j = (scores[t] >= min_hip_conf) & (~np.isnan(pts[t, :, 0])) & (~np.isnan(pts[t, :, 1]))
            if np.any(valid_j):
                raw_roots[t] = np.average(pts[t, valid_j], axis=0, weights=scores[t, valid_j])

    # Interpolate missing root frames and apply Savitzky-Golay smoothing
    root_df = pd.DataFrame(raw_roots, columns=['rx', 'ry'])
    root_df = root_df.interpolate(method='linear', limit_direction='both').ffill().bfill().fillna(0.0)
    roots = root_df.to_numpy(dtype=np.float64)

    if T >= 7:
        roots[:, 0] = signal.savgol_filter(roots[:, 0], window_length=7, polyorder=2)
        roots[:, 1] = signal.savgol_filter(roots[:, 1], window_length=7, polyorder=2)

    pts_centered = pts - roots[:, None, :]
    return pts_centered, roots


def _estimate_scale_90p(pts_centered, scores, min_conf=0.25):
    """
    Estimates the orthographic scale factor s using the 90th percentile of
    reference lower-limb bones (standing extension), avoiding foreshortening bias
    from deep squat frames in the front view.
    """
    lengths = []
    for a, b in REF_BONES:
        valid = (
            (scores[:, a] >= min_conf) &
            (scores[:, b] >= min_conf) &
            (~np.isnan(pts_centered[:, a, 0])) &
            (~np.isnan(pts_centered[:, b, 0]))
        )
        if np.any(valid):
            diff = pts_centered[valid, a, :] - pts_centered[valid, b, :]
            seg_len = np.linalg.norm(diff, axis=-1)
            seg_len = seg_len[seg_len > 1.0]
            if len(seg_len) > 0:
                lengths.append(np.percentile(seg_len, 90))

    if len(lengths) == 0:
        return 1.0
    ref_len = float(np.median(lengths))
    return 1.0 / max(ref_len, 1e-6)


def _interpolate_normalized_2d(pts_norm, valid_mask):
    """
    Linearly interpolates missing 2-D normalized coordinates per joint across time
    to construct a clean initial 3-D seed X_0.
    """
    T, J, _ = pts_norm.shape
    pts_interp = pts_norm.copy()
    for j in range(J):
        for d in range(2):
            col = pts_interp[:, j, d].copy()
            col[~valid_mask[:, j]] = np.nan
            s = pd.Series(col).interpolate(method='linear', limit_direction='both').ffill().bfill().fillna(0.0)
            pts_interp[:, j, d] = s.to_numpy()
    return pts_interp


def _build_jacobian_sparsity(T, J, pairs):
    """
    Builds the exact CSR Jacobian sparsity matrix for least_squares:
        n_res = 4*T*J + 3*(T-2)*J + T*P
        n_par = 3*T*J
    """
    P = len(pairs)
    n_reproj = 4 * T * J
    n_temp   = 3 * max(0, T - 2) * J
    n_bone   = T * P
    n_res    = n_reproj + n_temp + n_bone
    n_par    = 3 * T * J

    sparsity = lil_matrix((n_res, n_par), dtype=int)

    # 1. Reprojection blocks (4 * T * J rows)
    idx_TJ = np.arange(T * J)
    # Front X -> X^x (3*k + 0)
    sparsity[idx_TJ, 3 * idx_TJ + 0] = 1
    # Front Y -> X^y (3*k + 1)
    sparsity[T * J + idx_TJ, 3 * idx_TJ + 1] = 1
    # Side Z  -> X^z (3*k + 2)
    sparsity[2 * T * J + idx_TJ, 3 * idx_TJ + 2] = 1
    # Side Y  -> X^y (3*k + 1)
    sparsity[3 * T * J + idx_TJ, 3 * idx_TJ + 1] = 1

    # 2. Temporal smoothness block (3 * (T - 2) * J rows)
    if T >= 3:
        row_offset = n_reproj
        t_grid, j_grid, d_grid = np.meshgrid(
            np.arange(2, T), np.arange(J), np.arange(3), indexing='ij'
        )
        rows = row_offset + np.arange(n_temp)
        col_t0 = ((t_grid * J + j_grid) * 3 + d_grid).ravel()
        col_t1 = (((t_grid - 1) * J + j_grid) * 3 + d_grid).ravel()
        col_t2 = (((t_grid - 2) * J + j_grid) * 3 + d_grid).ravel()
        sparsity[rows, col_t0] = 1
        sparsity[rows, col_t1] = 1
        sparsity[rows, col_t2] = 1

    # 3. Bone-length prior block (T * P rows)
    row_offset = n_reproj + n_temp
    for b_idx, (p1, p2) in enumerate(pairs):
        t_arr = np.arange(T)
        rows = row_offset + t_arr * P + b_idx
        for d in range(3):
            sparsity[rows, (t_arr * J + p1) * 3 + d] = 1
            sparsity[rows, (t_arr * J + p2) * 3 + d] = 1

    return sparsity.tocsr()


def fuse_3d_pose_map(
    front_df,
    side_df,
    side_lag_frames=0,
    bone_weight=10.0,
    temporal_weight=1.0,
    robust_scale=0.05,
    n_em_iters=3,
    min_valid_conf=0.05,
    verbose=True
):
    """
    Fuses synchronized Front and Side 2-D keypoint DataFrames into a 3-D skeletal
    sequence using Batch EM and sparse Trust-Region Reflective MAP optimization.

    Returns:
        dict containing:
            - 'fused_3d_pts': (T, 17, 3) in front-camera pixel units (root-centered)
            - 'fused_scores': (T, 17) triangulation confidence in [0, 1]
            - 'detect_scores': (T, 17) noisy-OR detection probability in [0, 1]
            - 'final_bones': (16,) global 3D bone lengths in front-camera pixel units
            - 'roots_front': (T, 2) smoothed pelvic root in front pixel coordinates
            - 'roots_side': (T, 2) smoothed pelvic root in side pixel coordinates
            - 'scale_front': float s_f
            - 'scale_side': float s_s
            - 'front_pts_raw': (T, 17, 2) aligned raw front keypoints
            - 'side_pts_raw': (T, 17, 2) aligned raw side keypoints
    """
    # 1. Temporal Alignment
    f_sync, s_sync = align_keypoint_streams(front_df, side_df, side_lag_frames)
    T = len(f_sync)
    J = len(KEYPOINT_NAMES)
    P = len(SKELETON_PAIRS)

    if T < 5:
        raise ValueError(f"Sequence too short after synchronization (T={T} < 5 frames).")

    pts_f_raw, conf_f = _df_to_arrays(f_sync)
    pts_s_raw, conf_s = _df_to_arrays(s_sync)

    # 2. Root Centering & Scale Normalization (90th percentile standing reference)
    pts_f_cent, roots_f = _center_and_smooth_root(pts_f_raw, conf_f)
    pts_s_cent, roots_s = _center_and_smooth_root(pts_s_raw, conf_s)

    s_f = _estimate_scale_90p(pts_f_cent, conf_f)
    s_s = _estimate_scale_90p(pts_s_cent, conf_s)

    pts_f_norm = pts_f_cent * s_f
    pts_s_norm = pts_s_cent * s_s

    # 3. View-Specific Validity Masks & Confidence Weights
    valid_f = (~np.isnan(pts_f_norm[:, :, 0])) & (~np.isnan(pts_f_norm[:, :, 1])) & (conf_f > min_valid_conf)
    valid_s = (~np.isnan(pts_s_norm[:, :, 0])) & (~np.isnan(pts_s_norm[:, :, 1])) & (conf_s > min_valid_conf)

    w_f = np.where(valid_f, conf_f, 0.0)
    w_s = np.where(valid_s, conf_s, 0.0)

    # Replace NaNs in normalized observation arrays with 0.0 (residual weight is 0.0 anyway)
    u_f = np.where(valid_f, pts_f_norm[:, :, 0], 0.0)
    v_f = np.where(valid_f, pts_f_norm[:, :, 1], 0.0)
    u_s = np.where(valid_s, pts_s_norm[:, :, 0], 0.0)
    v_s = np.where(valid_s, pts_s_norm[:, :, 1], 0.0)

    # 4. Initialize 3-D Seed X_0 and 3-D Bone Lengths L^(0)
    f_interp = _interpolate_normalized_2d(pts_f_norm, valid_f)
    s_interp = _interpolate_normalized_2d(pts_s_norm, valid_s)

    X_init = np.zeros((T, J, 3), dtype=np.float64)
    X_init[:, :, 0] = f_interp[:, :, 0]  # X from Front u_f
    X_init[:, :, 2] = s_interp[:, :, 0]  # Z from Side u_s
    # Confidence-weighted vertical Y blend
    X_init[:, :, 1] = (
        w_f * f_interp[:, :, 1] + w_s * s_interp[:, :, 1]
    ) / (w_f + w_s + 1e-8)
    both_unseen = (w_f + w_s) < 1e-6
    X_init[:, :, 1][both_unseen] = 0.5 * (f_interp[:, :, 1][both_unseen] + s_interp[:, :, 1][both_unseen])

    # Seed 3-D bone lengths L_b^(0) using full 3-D Euclidean norm
    L_bones = np.zeros(P, dtype=np.float64)
    for b_idx, (p1, p2) in enumerate(SKELETON_PAIRS):
        valid_bone_t = (valid_f[:, p1] | valid_s[:, p1]) & (valid_f[:, p2] | valid_s[:, p2])
        dists_3d = np.linalg.norm(X_init[:, p1, :] - X_init[:, p2, :], axis=-1)
        if np.any(valid_bone_t):
            L_bones[b_idx] = float(np.median(dists_3d[valid_bone_t]))
        else:
            L_bones[b_idx] = float(np.median(dists_3d))
        L_bones[b_idx] = max(L_bones[b_idx], 0.05)

    # Precompute square-root weights and sparsity matrix
    sqrt_wf = np.sqrt(w_f).ravel()
    sqrt_ws = np.sqrt(w_s).ravel()
    sqrt_temp = np.sqrt(temporal_weight)
    sqrt_bone = np.sqrt(bone_weight)

    p1_idx = np.array([p[0] for p in SKELETON_PAIRS], dtype=np.int32)
    p2_idx = np.array([p[1] for p in SKELETON_PAIRS], dtype=np.int32)

    if verbose:
        print(f"Building sparse Jacobian pattern for T={T} frames ({3*T*J} variables)...")
    jac_sparsity = _build_jacobian_sparsity(T, J, SKELETON_PAIRS)

    # 5. Batch EM Optimization Loop
    X_curr = X_init.copy()

    for em_iter in range(n_em_iters):
        t0 = time.time()

        def residual_fn(x_flat):
            X = x_flat.reshape((T, J, 3))

            # Reprojection residuals (view-specific weights)
            res_fx = sqrt_wf * (X[:, :, 0].ravel() - u_f.ravel())
            res_fy = sqrt_wf * (X[:, :, 1].ravel() - v_f.ravel())
            res_sz = sqrt_ws * (X[:, :, 2].ravel() - u_s.ravel())
            res_sy = sqrt_ws * (X[:, :, 1].ravel() - v_s.ravel())

            # Temporal smoothness (second finite difference)
            res_temp = sqrt_temp * (X[2:] - 2.0 * X[1:-1] + X[:-2]).ravel()

            # Anatomical 3-D bone-length residuals
            bone_vecs = X[:, p1_idx, :] - X[:, p2_idx, :]  # (T, P, 3)
            bone_dists = np.sqrt(np.sum(bone_vecs**2, axis=-1) + 1e-12)  # (T, P)
            res_bone = sqrt_bone * (bone_dists - L_bones[None, :]).ravel()

            return np.concatenate([res_fx, res_fy, res_sz, res_sy, res_temp, res_bone])

        opt_res = least_squares(
            residual_fn,
            X_curr.ravel(),
            method='trf',
            loss='soft_l1',
            f_scale=robust_scale,
            x_scale='jac',
            jac_sparsity=jac_sparsity,
            max_nfev=50,
            verbose=0
        )

        X_curr = opt_res.x.reshape((T, J, 3))

        # M-step: Update global bone lengths using median over all frames
        new_L_bones = np.zeros_like(L_bones)
        for b_idx, (p1, p2) in enumerate(SKELETON_PAIRS):
            dists = np.linalg.norm(X_curr[:, p1, :] - X_curr[:, p2, :], axis=-1)
            new_L_bones[b_idx] = float(np.median(dists))

        max_delta_L = float(np.max(np.abs(new_L_bones - L_bones)))
        L_bones = new_L_bones

        if verbose:
            print(
                f"  [EM Iter {em_iter+1}/{n_em_iters}] "
                f"Cost: {opt_res.cost:.4f} | max |ΔL|: {max_delta_L:.5f} | "
                f"Time: {time.time() - t0:.2f}s"
            )

    # 6. Confidence Fusion & Inverse Scaling to Front-Camera Pixel Units
    fused_scores  = 0.5 * (w_f + w_s)
    detect_scores = w_f + w_s - (w_f * w_s)

    fused_3d_pts = X_curr / s_f
    final_bones  = L_bones / s_f

    return {
        "fused_3d_pts": fused_3d_pts,
        "fused_scores": fused_scores,
        "detect_scores": detect_scores,
        "final_bones": final_bones,
        "roots_front": roots_f,
        "roots_side": roots_s,
        "scale_front": s_f,
        "scale_side": s_s,
        "front_pts_raw": pts_f_raw,
        "side_pts_raw": pts_s_raw,
    }

In [ ]:
# @title Formatting, Storage & Interactive Visualization Utilities

def build_3d_pose_dataframe(map_result):
    """
    Converts the 3-D MAP result into a flat, tabular DataFrame suitable for Parquet storage.
    Columns: frame, <joint>_x, <joint>_y, <joint>_z, <joint>_score, <joint>_detect_prob,
             plus front/side root coordinates for exact 2-D reprojection.
    """
    pts_3d = map_result["fused_3d_pts"]
    scores = map_result["fused_scores"]
    detect = map_result["detect_scores"]
    roots_f = map_result["roots_front"]
    roots_s = map_result["roots_side"]

    T = pts_3d.shape[0]
    data = {"frame": np.arange(T, dtype=np.int32)}

    for j, name in enumerate(KEYPOINT_NAMES):
        data[f"{name}_x"] = pts_3d[:, j, 0]
        data[f"{name}_y"] = pts_3d[:, j, 1]
        data[f"{name}_z"] = pts_3d[:, j, 2]
        data[f"{name}_score"] = scores[:, j]
        data[f"{name}_detect_prob"] = detect[:, j]

    data["root_front_x"] = roots_f[:, 0]
    data["root_front_y"] = roots_f[:, 1]
    data["root_side_x"]  = roots_s[:, 0]
    data["root_side_y"]  = roots_s[:, 1]

    return pd.DataFrame(data)


def plot_interactive_3d_and_reprojection(map_result, frame_step=5):
    """
    Renders an interactive 3-panel Plotly viewer with a frame slider:
      1. Front View 2-D: Raw YOLO (grey) vs. MAP Reprojected (cyan)
      2. Side View 2-D:  Raw YOLO (grey) vs. MAP Reprojected (cyan)
      3. 3-D Root-Centered Skeleton (X, Z, -Y upright)
    """
    pts_3d = map_result["fused_3d_pts"]
    roots_f = map_result["roots_front"]
    roots_s = map_result["roots_side"]
    s_f = map_result["scale_front"]
    s_s = map_result["scale_side"]
    raw_f = map_result["front_pts_raw"]
    raw_s = map_result["side_pts_raw"]

    rho = s_f / s_s  # Scale ratio to map front-pixel 3D units back to side-pixel units
    T = pts_3d.shape[0]
    frames_to_plot = list(range(0, T, max(1, frame_step)))

    # Reproject 3-D points into 2-D front and side pixel coordinates
    reproj_f = np.zeros((T, 17, 2), dtype=np.float64)
    reproj_f[:, :, 0] = pts_3d[:, :, 0] + roots_f[:, None, 0]
    reproj_f[:, :, 1] = pts_3d[:, :, 1] + roots_f[:, None, 1]

    reproj_s = np.zeros((T, 17, 2), dtype=np.float64)
    reproj_s[:, :, 0] = pts_3d[:, :, 2] * rho + roots_s[:, None, 0]
    reproj_s[:, :, 1] = pts_3d[:, :, 1] * rho + roots_s[:, None, 1]

    def _get_skeleton_lines_2d(pts_frame):
        xs, ys = [], []
        for p1, p2 in SKELETON_PAIRS:
            if not (np.any(np.isnan(pts_frame[p1])) or np.any(np.isnan(pts_frame[p2]))):
                xs.extend([pts_frame[p1, 0], pts_frame[p2, 0], None])
                ys.extend([pts_frame[p1, 1], pts_frame[p2, 1], None])
        return xs, ys

    def _get_skeleton_lines_3d(pts_frame):
        xs, ys, zs = [], [], []
        for p1, p2 in SKELETON_PAIRS:
            xs.extend([pts_frame[p1, 0], pts_frame[p2, 0], None])
            # Map depth Z to horizontal Y-axis and -Y (upward) to vertical Z-axis in 3D plot
            ys.extend([pts_frame[p1, 2], pts_frame[p2, 2], None])
            zs.extend([-pts_frame[p1, 1], -pts_frame[p2, 1], None])
        return xs, ys, zs

    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "scene"}]],
        subplot_titles=("Front View (Grey: Raw, Cyan: MAP)", "Side View (Grey: Raw, Cyan: MAP)", "3-D Fused Skeleton")
    )

    t0 = frames_to_plot[0]
    rf_x, rf_y = _get_skeleton_lines_2d(raw_f[t0])
    pf_x, pf_y = _get_skeleton_lines_2d(reproj_f[t0])
    rs_x, rs_y = _get_skeleton_lines_2d(raw_s[t0])
    ps_x, ps_y = _get_skeleton_lines_2d(reproj_s[t0])
    p3_x, p3_y, p3_z = _get_skeleton_lines_3d(pts_3d[t0])

    # Traces 0, 1: Front Raw & MAP
    fig.add_trace(go.Scatter(x=rf_x, y=rf_y, mode="lines+markers", name="Raw Front", line=dict(color="grey", width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=pf_x, y=pf_y, mode="lines+markers", name="MAP Front", line=dict(color="cyan", width=3)), row=1, col=1)
    # Traces 2, 3: Side Raw & MAP
    fig.add_trace(go.Scatter(x=rs_x, y=rs_y, mode="lines+markers", name="Raw Side", line=dict(color="grey", width=2)), row=1, col=2)
    fig.add_trace(go.Scatter(x=ps_x, y=ps_y, mode="lines+markers", name="MAP Side", line=dict(color="cyan", width=3)), row=1, col=2)
    # Trace 4: 3-D Skeleton
    fig.add_trace(go.Scatter3d(x=p3_x, y=p3_y, z=p3_z, mode="lines+markers", name="3D Pose",
                               line=dict(color="#00cc96", width=5), marker=dict(size=4, color="#f79646")), row=1, col=3)

    # Build animation frames
    plotly_frames = []
    for t in frames_to_plot:
        rf_x, rf_y = _get_skeleton_lines_2d(raw_f[t])
        pf_x, pf_y = _get_skeleton_lines_2d(reproj_f[t])
        rs_x, rs_y = _get_skeleton_lines_2d(raw_s[t])
        ps_x, ps_y = _get_skeleton_lines_2d(reproj_s[t])
        p3_x, p3_y, p3_z = _get_skeleton_lines_3d(pts_3d[t])

        plotly_frames.append(go.Frame(
            name=str(t),
            data=[
                go.Scatter(x=rf_x, y=rf_y),
                go.Scatter(x=pf_x, y=pf_y),
                go.Scatter(x=rs_x, y=rs_y),
                go.Scatter(x=ps_x, y=ps_y),
                go.Scatter3d(x=p3_x, y=p3_y, z=p3_z)
            ]
        ))

    fig.frames = plotly_frames

    # Invert 2-D Y-axes so pixel (0,0) is at top-left
    fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1, row=1, col=1)
    fig.update_yaxes(autorange="reversed", scaleanchor="x2", scaleratio=1, row=1, col=2)

    max_range = float(np.nanpercentile(np.abs(pts_3d), 99)) * 1.2
    fig.update_layout(
        template="plotly_dark",
        height=600,
        scene=dict(
            xaxis=dict(title="X (Lateral)", range=[-max_range, max_range]),
            yaxis=dict(title="Z (Depth)", range=[-max_range, max_range]),
            zaxis=dict(title="-Y (Vertical)", range=[-max_range, max_range]),
            aspectmode="cube"
        ),
        sliders=[dict(
            steps=[dict(method="animate", args=[[str(t)], dict(mode="immediate", frame=dict(duration=0, redraw=True))], label=str(t))
                   for t in frames_to_plot],
            currentvalue=dict(prefix="Frame: ")
        )]
    )
    fig.show()


def run_3d_pose_pipeline(
    entry_id,
    store,
    bone_weight=10.0,
    temporal_weight=1.0,
    robust_scale=0.05,
    n_em_iters=3,
    plot=True,
    save_to_store=True
):
    """
    End-to-end execution of the 3-D MAP Pose Estimation for a single experiment entry.
    Loads `front_video_keypoints`, `side_video_keypoints`, and `video_sync_side_lag_frames`
    from OpenDatasetStore, runs the MAP solver, and saves `fused_3d_keypoints`.
    """
    entry_type = "experiments"
    print(f"=== Running 3-D MAP Pose Estimation for {entry_id} ===")

    # 1. Retrieve Entry Metadata & Processed 2-D Keypoint DataFrames
    entries_df = store.list_entries(entry_type=entry_type)
    row = entries_df[entries_df["entry_id"] == entry_id]
    if row.empty:
        print(f"Error: Entry {entry_id} not found in store.")
        return None, None

    side_lag = row.iloc[0].get("video_sync_side_lag_frames", np.nan)
    if pd.isna(side_lag):
        print(f"Error: `video_sync_side_lag_frames` is missing for {entry_id}. Run 03. Video Sync first.")
        return None, None

    try:
        front_df = store.get_processed_data(entry_type=entry_type, entry_id=entry_id, tag="front_video_keypoints")
        side_df  = store.get_processed_data(entry_type=entry_type, entry_id=entry_id, tag="side_video_keypoints")
    except Exception as e:
        print(f"Error loading 2-D keypoint parquet files for {entry_id}: {e}")
        return None, None

    if front_df is None or side_df is None or front_df.empty or side_df.empty:
        print(f"Error: Empty 2-D keypoint DataFrames for {entry_id}.")
        return None, None

    # 2. Run Bayesian MAP 3-D Fusion
    map_result = fuse_3d_pose_map(
        front_df=front_df,
        side_df=side_df,
        side_lag_frames=int(side_lag),
        bone_weight=bone_weight,
        temporal_weight=temporal_weight,
        robust_scale=robust_scale,
        n_em_iters=n_em_iters,
        verbose=True
    )

    pose_3d_df = build_3d_pose_dataframe(map_result)

    # 3. Diagnostic Visualization
    if plot:
        plot_interactive_3d_and_reprojection(map_result, frame_step=5)

    # 4. Save Back to OpenDatasetStore
    if save_to_store:
        bone_lengths_dict = {
            name: round(float(val), 3)
            for name, val in zip(BONE_NAMES, map_result["final_bones"])
        }
        with store_lock:
            store.add_processed_data(
                entry_type=entry_type,
                entry_id=entry_id,
                tag="fused_3d_keypoints",
                df=pose_3d_df,
                metadata={
                    "scale_front": float(map_result["scale_front"]),
                    "scale_side": float(map_result["scale_side"]),
                    "bone_lengths_front_px": bone_lengths_dict,
                    "bone_weight": float(bone_weight),
                    "temporal_weight": float(temporal_weight),
                    "robust_scale": float(robust_scale)
                }
            )
            print(f"✓ Saved `fused_3d_keypoints` ({len(pose_3d_df)} frames) and bone metadata for {entry_id}")

    return pose_3d_df, map_result

## **🚀 Run 3-D Pose Estimation**

In [ ]:
# @title Single Experiment Run & Interactive 3-D Viewer
target_entry_id = "experiments_0001" #@param {type:"string"}
bone_weight = 10.0 #@param {type:"number"}
temporal_weight = 1.0 #@param {type:"number"}
robust_scale = 0.05 #@param {type:"number"}
n_em_iters = 3 #@param {type:"integer"}

pose_3d_df, map_result = run_3d_pose_pipeline(
    entry_id=target_entry_id,
    store=store,
    bone_weight=bone_weight,
    temporal_weight=temporal_weight,
    robust_scale=robust_scale,
    n_em_iters=n_em_iters,
    plot=True,
    save_to_store=True
)

In [ ]:
# @title Batch Run Across All Synchronized Experiments
from concurrent.futures import ThreadPoolExecutor, as_completed

# Automatically filter all experiments that have a valid `video_sync_side_lag_frames`
all_entries = store.list_entries(entry_type="experiments")
synced_entries = all_entries[all_entries["video_sync_side_lag_frames"].notna()]["entry_id"].tolist()

print(f"Found {len(synced_entries)} synchronized experiments ready for 3-D MAP fusion: {synced_entries}")

def _worker_3d(exp_id):
    df_3d, res = run_3d_pose_pipeline(
        entry_id=exp_id,
        store=store,
        bone_weight=10.0,
        temporal_weight=1.0,
        robust_scale=0.05,
        n_em_iters=3,
        plot=False,
        save_to_store=True
    )
    return exp_id, (df_3d is not None)

MAX_WORKERS = 3
t_start = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_worker_3d, exp_id): exp_id for exp_id in synced_entries}
    for future in as_completed(futures):
        exp_id = futures[future]
        try:
            _, ok = future.result()
            status = "SUCCESS" if ok else "SKIPPED/FAILED"
            print(f"[{status}] {exp_id}")
        except Exception as e:
            print(f"[ERROR] {exp_id}: {e}")

print(f"\nBatch 3-D Pose Estimation completed in {time.time() - t_start:.1f}s!")